# Generative Models for Thermodynamic Path Optimization

**Key Insight**: If we can *predict* thermodynamic length with NTM, can we *optimize* it by generating intermediate molecules that shorten the path?

## Motivation

Current FEP practice:
- Given molecules A and B, run A → B transformation directly
- If thermodynamic length is large, convergence is slow/expensive

**Proposed approach**:
- Generate intermediate molecule(s) I such that:
  - A → I → B has shorter total thermodynamic length than A → B
  - I is synthetically accessible (or at least chemically valid)

$$\mathcal{L}(A \to I) + \mathcal{L}(I \to B) < \mathcal{L}(A \to B)$$

This is **geodesic refinement** — finding waypoints that reduce total path cost.

---
## Theoretical Foundation

### Why Intermediates Can Help

In Riemannian geometry, the triangle inequality states:

$$d(A, B) \leq d(A, I) + d(I, B)$$

But this is for the *geodesic* distance. In practice, FEP doesn't follow geodesics — it uses linear interpolation in λ-space. The actual thermodynamic length can be:

$$\mathcal{L}_{\text{linear}}(A \to B) > \mathcal{L}_{\text{linear}}(A \to I) + \mathcal{L}_{\text{linear}}(I \to B)$$

This happens when the direct path crosses a **high-energy barrier** that intermediates can circumvent.

### Analogy: Mountain Climbing

| Direct Path | Via Intermediate |
|-------------|------------------|
| Straight line over mountain peak | Path through mountain pass |
| Short Euclidean distance | Longer Euclidean distance |
| High energy cost | Lower energy cost |

### Formal Problem Statement

Given:
- Source molecule A with embedding $\mathbf{h}_A$
- Target molecule B with embedding $\mathbf{h}_B$
- Learned metric tensor $\mathbf{M}$
- NTM distance function $d_M(\cdot, \cdot)$

Find:
- Intermediate molecule(s) $I_1, I_2, \ldots, I_k$ such that:

$$\min_{I_1, \ldots, I_k} \sum_{j=0}^{k} d_M(I_j, I_{j+1})$$

where $I_0 = A$ and $I_{k+1} = B$.

**Constraints**:
1. Each $I_j$ must be a valid molecule
2. Each $I_j$ should be synthetically accessible (optional)
3. $k$ should be small (practical constraint)

---
## Clarification: NTM Distance vs Thermodynamic Length

### The Subtle Distinction

There are **two different quantities** at play:

| Quantity | Symbol | Definition | Triangle Inequality? |
|----------|--------|------------|---------------------|
| **NTM Distance** | $d_M(A,B)$ | $\sqrt{\Delta h^T M \Delta h}$ | Yes (it's a metric) |
| **Thermodynamic Length** | $\mathcal{L}(A \to B)$ | $\int_0^1 \sqrt{\text{Var}(\partial U/\partial \lambda)} d\lambda$ | **No** (path-dependent) |

### Why Intermediates Can Help

NTM distance $d_M$ is a **surrogate** for thermodynamic length $\mathcal{L}$. The surrogate satisfies the triangle inequality, but the true quantity may not:

$$d_M(A,I) + d_M(I,B) \geq d_M(A,B) \quad \text{(always true)}$$

$$\mathcal{L}(A \to I) + \mathcal{L}(I \to B) \lessgtr \mathcal{L}(A \to B) \quad \text{(depends on path)}$$

### The Key Assumption

We assume NTM is trained such that:

$$d_M(A,B) \approx \mathcal{L}_{\text{linear}}(A \to B)$$

where $\mathcal{L}_{\text{linear}}$ is the thermodynamic length along the **linear λ-interpolation** path (what FEP actually uses).

If this holds, then finding an intermediate I where:

$$d_M(A,I) + d_M(I,B) \approx d_M(A,B)$$

(equality, since $d_M$ is a metric) corresponds to:

$$\mathcal{L}_{\text{linear}}(A \to I) + \mathcal{L}_{\text{linear}}(I \to B) < \mathcal{L}_{\text{linear}}(A \to B)$$

because the **two shorter linear paths** may avoid barriers that the **one long linear path** crosses.

### When Does This Work?

Intermediates help when:
1. The direct A→B linear path crosses a high-energy barrier
2. There exists a molecule I such that neither A→I nor I→B crosses that barrier
3. The NTM has learned to predict this barrier-crossing difficulty

Intermediates do NOT help when:
- A and B are already similar (low $d_M$)
- The transformation is uniformly difficult (no barrier to circumvent)
- No valid molecule exists in the "valley" region

---
## The Geodesic Shortening Principle

### Continuous Formulation

In the continuous limit, we seek a path $\gamma: [0,1] \to \mathcal{M}$ (molecular manifold) that minimizes:

$$\mathcal{L}[\gamma] = \int_0^1 \sqrt{\dot{\gamma}(t)^T \mathbf{M}(\gamma(t)) \dot{\gamma}(t)} \, dt$$

The solution is the **geodesic** — but geodesics live in continuous embedding space, not discrete molecular space.

### Discrete Approximation

We approximate the geodesic with a sequence of real molecules:

$$A = M_0 \to M_1 \to M_2 \to \cdots \to M_k \to M_{k+1} = B$$

Each $M_i$ is a valid molecule whose embedding $\mathbf{h}_{M_i}$ lies near the continuous geodesic.

### Why This Works

1. **Barrier avoidance**: Intermediates can route around high-curvature regions
2. **Smaller perturbations**: Each step is a smaller chemical change
3. **Better overlap**: Adjacent λ-windows have better phase space overlap

### Connection to λ-Scheduling

Traditional FEP uses fixed λ-windows: $\lambda \in \{0, 0.1, 0.2, \ldots, 1.0\}$

With intermediates, we effectively create a **multi-leg transformation**:
- Leg 1: A → I (with its own λ-schedule)
- Leg 2: I → B (with its own λ-schedule)

Total free energy: $\Delta G_{A \to B} = \Delta G_{A \to I} + \Delta G_{I \to B}$ (thermodynamic cycle)

---
## NTM Components as Steering Signals for Generation

The key insight is that NTM doesn't just produce a scalar "difficulty" — it provides **rich geometric information** that can directly steer molecular generation.

### Decomposing Transformation Difficulty

The NTM distance between molecules A and B is:

$$d_M(A, B) = \sqrt{(\mathbf{h}_B - \mathbf{h}_A)^T \mathbf{M} (\mathbf{h}_B - \mathbf{h}_A)}$$

This can be decomposed into **directional components** via the metric tensor's eigendecomposition:

$$\mathbf{M} = \sum_{i=1}^{d} \lambda_i \mathbf{v}_i \mathbf{v}_i^T$$

where $\lambda_i$ are eigenvalues and $\mathbf{v}_i$ are eigenvectors.

The difficulty along each eigendirection is:

$$d_i^2 = \lambda_i \cdot (\mathbf{v}_i^T \Delta \mathbf{h})^2$$

where $\Delta \mathbf{h} = \mathbf{h}_B - \mathbf{h}_A$.

### Interpretation of Eigendirections

| Eigenvalue | Meaning | Generation Strategy |
|------------|---------|---------------------|
| **Large $\lambda_i$** | "Hard" direction — small changes cause large difficulty | Avoid moving along $\mathbf{v}_i$ |
| **Small $\lambda_i$** | "Easy" direction — can move freely | Prefer moving along $\mathbf{v}_i$ |
| **$\mathbf{v}_i^T \Delta \mathbf{h}$ large** | Transformation requires movement in direction $i$ | Must address this component |

### The Steering Principle

**Goal**: Generate intermediates that decompose a hard transformation into easier steps by:
1. Identifying which eigendirections contribute most to difficulty
2. Finding molecules that "turn the corner" — moving along easy directions first
3. Deferring movement along hard directions to later steps

Mathematically, for intermediate I:

$$\text{Steer toward } I \text{ such that: } \mathbf{v}_{\text{hard}}^T (\mathbf{h}_I - \mathbf{h}_A) \approx 0$$

This keeps the A → I transformation in the "easy subspace" while I → B handles the hard directions (or vice versa).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional

# =============================================================================
# Transformation Difficulty Decomposition
# =============================================================================

class TransformationDifficultyAnalyzer:
    """
    Decompose NTM transformation difficulty into directional components.
    
    This provides the steering signals for generative models.
    """
    
    def __init__(self, metric_tensor: torch.Tensor):
        """
        Args:
            metric_tensor: Learned metric M (d x d positive definite matrix)
        """
        self.M = metric_tensor
        
        # Eigendecomposition: M = V Λ V^T
        eigenvalues, eigenvectors = torch.linalg.eigh(metric_tensor)
        
        # Sort by eigenvalue (descending = hardest directions first)
        idx = torch.argsort(eigenvalues, descending=True)
        self.eigenvalues = eigenvalues[idx]
        self.eigenvectors = eigenvectors[:, idx]
        
        # Normalize eigenvalues for interpretability
        self.eigenvalue_weights = self.eigenvalues / self.eigenvalues.sum()
    
    def decompose_difficulty(
        self, 
        h_a: torch.Tensor, 
        h_b: torch.Tensor
    ) -> dict:
        """
        Decompose the A → B transformation difficulty by eigendirection.
        
        Returns:
            Dictionary with per-direction difficulty contributions
        """
        delta_h = h_b - h_a
        
        # Project onto each eigendirection
        projections = self.eigenvectors.T @ delta_h  # (d,)
        
        # Difficulty contribution from each direction: λ_i * (v_i · Δh)²
        difficulty_per_direction = self.eigenvalues * (projections ** 2)
        
        # Total difficulty
        total_difficulty = difficulty_per_direction.sum().sqrt()
        
        # Fraction of difficulty from each direction
        difficulty_fractions = difficulty_per_direction / difficulty_per_direction.sum()
        
        return {
            'total_difficulty': total_difficulty.item(),
            'projections': projections,
            'difficulty_per_direction': difficulty_per_direction,
            'difficulty_fractions': difficulty_fractions,
            'dominant_direction_idx': torch.argmax(difficulty_fractions).item(),
            'eigenvalues': self.eigenvalues,
            'eigenvectors': self.eigenvectors
        }
    
    def get_steering_gradient(
        self,
        h_a: torch.Tensor,
        h_b: torch.Tensor,
        h_current: torch.Tensor,
        temperature: float = 1.0
    ) -> torch.Tensor:
        """
        Compute steering gradient for generation.
        
        This gradient points toward embeddings that reduce path difficulty.
        
        The key insight: we want to move toward B, but preferentially
        along EASY directions (small eigenvalues) first.
        
        Args:
            h_a: Source embedding
            h_b: Target embedding  
            h_current: Current position in embedding space
            temperature: Controls sharpness of direction preference
        
        Returns:
            Steering gradient in embedding space
        """
        # Direction to target
        to_target = h_b - h_current
        
        # Weight directions inversely by eigenvalue (prefer easy directions)
        # Softmax over negative eigenvalues = prefer small eigenvalues
        direction_weights = torch.softmax(-self.eigenvalues / temperature, dim=0)
        
        # Project to_target onto eigenbasis
        projections = self.eigenvectors.T @ to_target
        
        # Reweight: amplify easy directions, suppress hard directions
        reweighted_projections = projections * direction_weights
        
        # Transform back to original space
        steering_gradient = self.eigenvectors @ reweighted_projections
        
        # Normalize to unit vector
        steering_gradient = steering_gradient / (steering_gradient.norm() + 1e-8)
        
        return steering_gradient
    
    def find_easy_subspace(self, threshold: float = 0.1) -> torch.Tensor:
        """
        Find the subspace of "easy" transformations.
        
        Returns projection matrix onto easy subspace.
        """
        # Easy directions: eigenvalues below threshold * max
        easy_mask = self.eigenvalues < threshold * self.eigenvalues.max()
        easy_vectors = self.eigenvectors[:, easy_mask]
        
        # Projection matrix onto easy subspace
        P_easy = easy_vectors @ easy_vectors.T
        
        return P_easy

# Demonstration
print("TransformationDifficultyAnalyzer defined.")
print("\nKey methods:")
print("  - decompose_difficulty(): Break down A→B difficulty by eigendirection")
print("  - get_steering_gradient(): Gradient that prefers easy directions")
print("  - find_easy_subspace(): Projection onto low-difficulty subspace")

---
## Gradient-Based Steering for Generative Models

### The Core Idea

Instead of just *scoring* generated molecules, we use NTM's geometric structure to *guide* the generation process. This is analogous to classifier-free guidance in diffusion models, but using transformation difficulty as the signal.

### Mathematical Framework

Let $G_\theta$ be a generative model (VAE decoder, diffusion model, etc.) that maps latent code $\mathbf{z}$ to molecular representation $\mathbf{x}$:

$$\mathbf{x} = G_\theta(\mathbf{z})$$

Let $\phi$ be the NTM encoder that maps molecules to embeddings:

$$\mathbf{h} = \phi(\mathbf{x})$$

The **steering objective** is to find $\mathbf{z}^*$ such that the generated molecule minimizes path difficulty:

$$\mathbf{z}^* = \arg\min_{\mathbf{z}} \left[ d_M(\mathbf{h}_A, \phi(G_\theta(\mathbf{z}))) + d_M(\phi(G_\theta(\mathbf{z})), \mathbf{h}_B) \right]$$

### Gradient Computation

The steering gradient through the full pipeline is:

$$\nabla_{\mathbf{z}} \mathcal{L} = \underbrace{\frac{\partial \mathcal{L}}{\partial \mathbf{h}}}_{\text{NTM gradient}} \cdot \underbrace{\frac{\partial \mathbf{h}}{\partial \mathbf{x}}}_{\text{Encoder Jacobian}} \cdot \underbrace{\frac{\partial \mathbf{x}}{\partial \mathbf{z}}}_{\text{Generator Jacobian}}$$

where $\mathcal{L} = d_M(A, I) + d_M(I, B)$.

### The NTM Gradient (Key Contribution)

The gradient of NTM distance with respect to the intermediate embedding is:

$$\frac{\partial d_M(A, I)}{\partial \mathbf{h}_I} = \frac{\mathbf{M}(\mathbf{h}_I - \mathbf{h}_A)}{d_M(A, I)}$$

This points **away from A** in the metric-weighted space. Combined with the gradient toward B:

$$\frac{\partial d_M(I, B)}{\partial \mathbf{h}_I} = \frac{\mathbf{M}(\mathbf{h}_I - \mathbf{h}_B)}{d_M(I, B)}$$

The total steering gradient is:

$$\nabla_{\mathbf{h}_I} \mathcal{L} = \frac{\mathbf{M}(\mathbf{h}_I - \mathbf{h}_A)}{d_M(A, I)} + \frac{\mathbf{M}(\mathbf{h}_I - \mathbf{h}_B)}{d_M(I, B)}$$

### Eigenspace Interpretation

Decomposing this gradient in the metric eigenbasis reveals which directions to move:

$$\nabla_{\mathbf{h}_I} \mathcal{L} = \sum_i \lambda_i \left[ \frac{(\mathbf{v}_i^T \Delta_{AI})}{d_M(A,I)} + \frac{(\mathbf{v}_i^T \Delta_{IB})}{d_M(I,B)} \right] \mathbf{v}_i$$

**Key insight**: Directions with large $\lambda_i$ contribute more to the gradient. The optimal intermediate lies where these high-$\lambda$ components cancel out.

In [ ]:
# =============================================================================
# Gradient-Based Steering Implementation
# =============================================================================

class NTMSteeringModule(nn.Module):
    """
    Module that computes steering gradients for generative models.
    
    This is the core component that translates NTM's geometric structure
    into actionable gradients for molecular generation.
    """
    
    def __init__(self, ntm_encoder: nn.Module, metric_tensor: torch.Tensor):
        """
        Args:
            ntm_encoder: Trained NTM encoder φ that maps molecules to embeddings
            metric_tensor: Learned metric M
        """
        super().__init__()
        self.encoder = ntm_encoder
        self.M = metric_tensor
        self.analyzer = TransformationDifficultyAnalyzer(metric_tensor)
    
    def compute_ntm_distance(self, h1: torch.Tensor, h2: torch.Tensor) -> torch.Tensor:
        """Compute NTM distance between two embeddings."""
        delta = h2 - h1
        return torch.sqrt(delta @ self.M @ delta + 1e-8)
    
    def compute_path_length(
        self, 
        h_a: torch.Tensor, 
        h_intermediate: torch.Tensor, 
        h_b: torch.Tensor
    ) -> torch.Tensor:
        """Total path length A → I → B."""
        return self.compute_ntm_distance(h_a, h_intermediate) + \
               self.compute_ntm_distance(h_intermediate, h_b)
    
    def compute_steering_loss(
        self,
        mol_a: torch.Tensor,
        mol_b: torch.Tensor,
        generated_intermediate: torch.Tensor,
        include_regularization: bool = True,
        reg_weight: float = 0.1
    ) -> Tuple[torch.Tensor, dict]:
        """
        Compute the steering loss for training/guiding a generative model.
        
        Args:
            mol_a, mol_b: Source and target molecules
            generated_intermediate: Generated intermediate (differentiable)
            include_regularization: Add regularization terms
            reg_weight: Weight for regularization
        
        Returns:
            loss: Scalar loss to minimize
            info: Dictionary with diagnostic information
        """
        # Encode all molecules
        h_a = self.encoder(mol_a)
        h_b = self.encoder(mol_b)
        h_i = self.encoder(generated_intermediate)
        
        # Direct path length (for comparison)
        d_direct = self.compute_ntm_distance(h_a, h_b)
        
        # Path via intermediate
        d_ai = self.compute_ntm_distance(h_a, h_i)
        d_ib = self.compute_ntm_distance(h_i, h_b)
        d_via = d_ai + d_ib
        
        # Primary loss: minimize path length via intermediate
        path_loss = d_via
        
        # Regularization: encourage intermediate to be "between" A and B
        # (not too close to either endpoint)
        if include_regularization:
            # Penalize if intermediate is too close to A or B
            min_distance = torch.min(d_ai, d_ib)
            balance_reg = -torch.log(min_distance + 0.1)  # Encourage min_distance > 0
            
            total_loss = path_loss + reg_weight * balance_reg
        else:
            balance_reg = torch.tensor(0.0)
            total_loss = path_loss
        
        # Compute improvement
        improvement = (d_direct - d_via) / d_direct
        
        info = {
            'd_direct': d_direct.item(),
            'd_via_intermediate': d_via.item(),
            'd_a_to_i': d_ai.item(),
            'd_i_to_b': d_ib.item(),
            'improvement': improvement.item(),
            'balance_reg': balance_reg.item() if include_regularization else 0.0
        }
        
        return total_loss, info
    
    def get_embedding_steering_gradient(
        self,
        h_a: torch.Tensor,
        h_b: torch.Tensor,
        h_current: torch.Tensor
    ) -> torch.Tensor:
        """
        Compute the analytical steering gradient in embedding space.
        
        This is the gradient of path length w.r.t. intermediate position.
        """
        h_current = h_current.clone().requires_grad_(True)
        
        d_ai = self.compute_ntm_distance(h_a, h_current)
        d_ib = self.compute_ntm_distance(h_current, h_b)
        path_length = d_ai + d_ib
        
        grad = torch.autograd.grad(path_length, h_current)[0]
        
        return grad
    
    def decompose_steering_gradient(
        self,
        h_a: torch.Tensor,
        h_b: torch.Tensor,
        h_current: torch.Tensor
    ) -> dict:
        """
        Decompose the steering gradient by metric eigendirection.
        
        This shows which "difficulty components" are driving the gradient.
        """
        grad = self.get_embedding_steering_gradient(h_a, h_b, h_current)
        
        # Project gradient onto eigenbasis
        grad_projections = self.analyzer.eigenvectors.T @ grad
        
        # Contribution from each eigendirection
        contributions = grad_projections.abs() * self.analyzer.eigenvalues.sqrt()
        
        return {
            'gradient': grad,
            'projections_per_eigendirection': grad_projections,
            'contribution_per_eigendirection': contributions,
            'dominant_eigendirection': torch.argmax(contributions).item(),
            'eigenvalues': self.analyzer.eigenvalues
        }

print("NTMSteeringModule defined.")
print("\nThis module provides:")
print("  - compute_steering_loss(): Differentiable loss for generator training")
print("  - get_embedding_steering_gradient(): Direct gradient in embedding space")
print("  - decompose_steering_gradient(): Per-eigendirection gradient analysis")

---
## Connecting Metric Eigenstructure to Generation Constraints

### The Anisotropic Difficulty Landscape

The metric tensor $\mathbf{M}$ defines an **anisotropic** difficulty landscape — some directions in molecular space are harder to traverse than others. This anisotropy is the key to effective steering.

### Eigenvalue Interpretation for Drug Discovery

| Eigendirection Type | Chemical Interpretation | Generation Strategy |
|---------------------|------------------------|---------------------|
| **High λ (hard)** | Core scaffold changes, charge modifications, ring system alterations | Avoid or defer to final step |
| **Medium λ** | Functional group swaps, heteroatom changes | Can traverse with care |
| **Low λ (easy)** | R-group variations, halogen swaps, chain length | Traverse freely |

### Constrained Generation in Easy Subspace

For the first intermediate, we can constrain generation to the **easy subspace**:

$$\mathbf{h}_I \in \text{span}\{\mathbf{v}_i : \lambda_i < \tau\}$$

where $\tau$ is a difficulty threshold.

This ensures the A → I transformation only involves "easy" chemical changes, deferring hard changes to I → B.

### The Optimal Intermediate Theorem

**Theorem**: For a transformation A → B with difficulty dominated by eigendirection $\mathbf{v}_k$ (large $\lambda_k$), the optimal single intermediate $I^*$ satisfies:

$$\mathbf{v}_k^T \mathbf{h}_{I^*} = \frac{1}{2}(\mathbf{v}_k^T \mathbf{h}_A + \mathbf{v}_k^T \mathbf{h}_B)$$

**Proof sketch**: The path length $d_M(A,I) + d_M(I,B)$ is minimized when I lies on the geodesic. For the dominant eigendirection, this means I should be at the midpoint along $\mathbf{v}_k$.

**Implication**: The steering gradient naturally pushes generated molecules toward this midpoint in hard directions, while allowing freedom in easy directions.

In [ ]:
# =============================================================================
# Visualization: Eigenstructure and Steering
# =============================================================================

def visualize_difficulty_decomposition():
    """
    Visualize how transformation difficulty decomposes by eigendirection,
    and how this informs steering.
    """
    np.random.seed(42)
    
    # Create a synthetic metric tensor with clear eigenstructure
    # 2D for visualization
    eigenvalues = np.array([4.0, 0.5])  # Hard and easy directions
    theta = np.pi / 6  # Rotate 30 degrees
    V = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta), np.cos(theta)]])
    M = V @ np.diag(eigenvalues) @ V.T
    
    # Define A and B
    h_A = np.array([-1.0, -0.5])
    h_B = np.array([1.0, 0.8])
    delta_h = h_B - h_A
    
    # Decompose difficulty
    projections = V.T @ delta_h
    difficulty_per_dir = eigenvalues * projections**2
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Plot 1: Embedding space with metric ellipse
    ax1 = axes[0]
    
    # Draw metric ellipse (unit difficulty contour)
    theta_range = np.linspace(0, 2*np.pi, 100)
    ellipse = np.zeros((100, 2))
    for i, t in enumerate(theta_range):
        direction = np.array([np.cos(t), np.sin(t)])
        # Scale by inverse sqrt of eigenvalue in that direction
        proj = V.T @ direction
        scale = 1.0 / np.sqrt(np.sum(eigenvalues * proj**2))
        ellipse[i] = scale * direction
    
    ax1.plot(ellipse[:, 0], ellipse[:, 1], 'gray', linestyle='--', label='Unit difficulty')
    
    # Draw eigenvectors
    for i, (ev, lam) in enumerate(zip(V.T, eigenvalues)):
        color = 'red' if lam > 1 else 'green'
        label = f'v{i+1} (λ={lam:.1f}, {"hard" if lam > 1 else "easy"})'
        ax1.arrow(0, 0, ev[0]*0.8, ev[1]*0.8, head_width=0.1, color=color, label=label)
    
    # Draw A, B, and transformation vector
    ax1.scatter(*h_A, s=150, c='blue', marker='o', zorder=5, label='A')
    ax1.scatter(*h_B, s=150, c='red', marker='s', zorder=5, label='B')
    ax1.arrow(h_A[0], h_A[1], delta_h[0]*0.9, delta_h[1]*0.9, 
              head_width=0.1, color='purple', linestyle='-', linewidth=2)
    
    # Optimal intermediate (midpoint in hard direction)
    h_I_optimal = h_A + 0.5 * delta_h  # Simplified: true midpoint
    ax1.scatter(*h_I_optimal, s=150, c='green', marker='^', zorder=5, label='I* (optimal)')
    
    ax1.set_xlim(-2, 2)
    ax1.set_ylim(-1.5, 1.5)
    ax1.set_xlabel('Embedding dim 1')
    ax1.set_ylabel('Embedding dim 2')
    ax1.set_title('Metric Eigenstructure')
    ax1.legend(loc='upper left', fontsize=8)
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Difficulty decomposition
    ax2 = axes[1]
    directions = ['Hard (v1)', 'Easy (v2)']
    colors = ['red', 'green']
    ax2.bar(directions, difficulty_per_dir, color=colors, edgecolor='black')
    ax2.set_ylabel('Difficulty Contribution')
    ax2.set_title('Difficulty by Eigendirection')
    
    for i, v in enumerate(difficulty_per_dir):
        ax2.text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=12)
    
    total = np.sqrt(difficulty_per_dir.sum())
    ax2.axhline(y=0, color='black', linewidth=0.5)
    ax2.set_ylim(0, max(difficulty_per_dir) * 1.3)
    ax2.text(0.5, max(difficulty_per_dir) * 1.15, f'Total d_M = {total:.2f}', 
             ha='center', transform=ax2.get_xaxis_transform(), fontsize=11)
    
    # Plot 3: Steering gradient field
    ax3 = axes[2]
    
    # Create grid
    x = np.linspace(-2, 2, 15)
    y = np.linspace(-1.5, 1.5, 12)
    X, Y = np.meshgrid(x, y)
    
    # Compute steering gradient at each point
    U = np.zeros_like(X)
    V_field = np.zeros_like(Y)
    
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            h_curr = np.array([X[i, j], Y[i, j]])
            
            # Gradient of path length A → curr → B
            d_ai = np.sqrt((h_curr - h_A) @ M @ (h_curr - h_A) + 1e-8)
            d_ib = np.sqrt((h_B - h_curr) @ M @ (h_B - h_curr) + 1e-8)
            
            grad = M @ (h_curr - h_A) / d_ai + M @ (h_curr - h_B) / d_ib
            grad = -grad  # Negative for descent
            
            # Normalize for visualization
            norm = np.linalg.norm(grad)
            if norm > 0.1:
                U[i, j] = grad[0] / norm
                V_field[i, j] = grad[1] / norm
    
    ax3.quiver(X, Y, U, V_field, alpha=0.6, color='gray')
    ax3.scatter(*h_A, s=150, c='blue', marker='o', zorder=5, label='A')
    ax3.scatter(*h_B, s=150, c='red', marker='s', zorder=5, label='B')
    ax3.scatter(*h_I_optimal, s=150, c='green', marker='^', zorder=5, label='I*')
    
    ax3.set_xlabel('Embedding dim 1')
    ax3.set_ylabel('Embedding dim 2')
    ax3.set_title('Steering Gradient Field')
    ax3.legend(loc='upper left', fontsize=8)
    ax3.set_aspect('equal')
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nInterpretation:")
    print(f"  - Hard direction (v1, λ=4.0): contributes {difficulty_per_dir[0]:.2f} to difficulty")
    print(f"  - Easy direction (v2, λ=0.5): contributes {difficulty_per_dir[1]:.2f} to difficulty")
    print(f"  - Steering gradient points toward optimal intermediate I*")
    print(f"  - Generator should follow gradient to find path-shortening molecules")

visualize_difficulty_decomposition()

---
## Integrating NTM Steering with Generative Architectures

### Architecture Overview

The NTM steering framework can be integrated with various generative models:

```
┌─────────────────────────────────────────────────────────────────┐
│                    NTM-Steered Generation                       │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ┌──────────┐    ┌──────────────┐    ┌──────────────────────┐  │
│  │ Latent z │───▶│  Generator   │───▶│ Generated Molecule I │  │
│  └──────────┘    │   G_θ(z)     │    └──────────┬───────────┘  │
│       ▲          └──────────────┘               │              │
│       │                                         ▼              │
│       │          ┌──────────────┐    ┌──────────────────────┐  │
│       │          │ NTM Encoder  │◀───│   Molecular Rep.     │  │
│       │          │    φ(x)      │    └──────────────────────┘  │
│       │          └──────┬───────┘                              │
│       │                 ▼                                      │
│       │          ┌──────────────┐                              │
│       │          │ Embedding h_I│                              │
│       │          └──────┬───────┘                              │
│       │                 │                                      │
│       │                 ▼                                      │
│       │    ┌────────────────────────────┐                      │
│       │    │  Steering Loss Computation │                      │
│       │    │  L = d_M(A,I) + d_M(I,B)   │                      │
│       │    │  + eigenspace constraints  │                      │
│       │    └────────────┬───────────────┘                      │
│       │                 │                                      │
│       │                 ▼                                      │
│       │    ┌────────────────────────────┐                      │
│       └────│   Backprop ∂L/∂z          │                      │
│            │   (Steering Gradient)      │                      │
│            └────────────────────────────┘                      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Three Integration Modes

| Mode | Description | When to Use |
|------|-------------|-------------|
| **Training-time** | Add steering loss to generator training objective | When training generator from scratch |
| **Inference-time** | Use steering gradient to guide sampling | With pretrained generators |
| **Latent optimization** | Directly optimize z to minimize path length | For specific A,B pairs |

### Mode 1: Training-Time Integration

Modify the generator's training objective:

$$\mathcal{L}_{total} = \mathcal{L}_{reconstruction} + \alpha \cdot \mathcal{L}_{steering}$$

where $\mathcal{L}_{steering}$ encourages generated molecules to be good intermediates for random (A, B) pairs from the training set.

### Mode 2: Inference-Time Guidance (Diffusion Models)

For diffusion models, modify the score function:

$$\tilde{s}_\theta(x_t, t) = s_\theta(x_t, t) - w \cdot \nabla_{x_t} [d_M(A, x_t) + d_M(x_t, B)]$$

This is **classifier-free guidance** but using NTM path length as the "classifier."

### Mode 3: Latent Space Optimization

Given a pretrained generator, optimize the latent code directly:

$$z^* = \arg\min_z \left[ d_M(A, \phi(G(z))) + d_M(\phi(G(z)), B) \right]$$

Solved via gradient descent on z.

In [ ]:
# =============================================================================
# Complete NTM-Steered Generator Implementation
# =============================================================================

class NTMSteeredGenerator(nn.Module):
    """
    A generative model steered by NTM transformation difficulty components.
    
    This integrates all the theoretical components:
    1. Difficulty decomposition via metric eigenstructure
    2. Steering gradients that prefer easy directions
    3. Eigenspace constraints for controlled generation
    """
    
    def __init__(
        self,
        generator: nn.Module,
        ntm_encoder: nn.Module,
        metric_tensor: torch.Tensor,
        latent_dim: int,
        steering_weight: float = 1.0,
        eigenspace_constraint_weight: float = 0.5
    ):
        """
        Args:
            generator: Pretrained molecular generator G(z) -> x
            ntm_encoder: NTM encoder φ(x) -> h
            metric_tensor: Learned metric M
            latent_dim: Dimension of latent space z
            steering_weight: Weight for path-shortening objective
            eigenspace_constraint_weight: Weight for eigenspace regularization
        """
        super().__init__()
        self.generator = generator
        self.ntm_encoder = ntm_encoder
        self.M = metric_tensor
        self.latent_dim = latent_dim
        self.steering_weight = steering_weight
        self.eigenspace_weight = eigenspace_constraint_weight
        
        # Eigendecomposition for steering
        eigenvalues, eigenvectors = torch.linalg.eigh(metric_tensor)
        idx = torch.argsort(eigenvalues, descending=True)
        self.register_buffer('eigenvalues', eigenvalues[idx])
        self.register_buffer('eigenvectors', eigenvectors[:, idx])
        
        # Identify hard vs easy subspaces
        threshold = 0.3 * self.eigenvalues.max()
        self.n_hard_directions = (self.eigenvalues > threshold).sum().item()
    
    def compute_ntm_distance(self, h1: torch.Tensor, h2: torch.Tensor) -> torch.Tensor:
        """NTM distance between embeddings."""
        delta = h2 - h1
        return torch.sqrt(torch.einsum('...i,ij,...j->...', delta, self.M, delta) + 1e-8)
    
    def decompose_transformation(
        self, 
        h_a: torch.Tensor, 
        h_b: torch.Tensor
    ) -> dict:
        """
        Decompose A→B transformation into difficulty components.
        
        Returns information about which eigendirections dominate.
        """
        delta = h_b - h_a
        
        # Project onto eigenbasis
        projections = torch.einsum('ij,...j->...i', self.eigenvectors.T, delta)
        
        # Per-direction difficulty: λ_i * (projection_i)²
        difficulty_components = self.eigenvalues * (projections ** 2)
        
        # Identify dominant directions
        total_difficulty_sq = difficulty_components.sum(dim=-1, keepdim=True)
        fractions = difficulty_components / (total_difficulty_sq + 1e-8)
        
        return {
            'projections': projections,
            'difficulty_components': difficulty_components,
            'difficulty_fractions': fractions,
            'total_difficulty': torch.sqrt(total_difficulty_sq.squeeze(-1)),
            'hard_direction_contribution': fractions[..., :self.n_hard_directions].sum(dim=-1)
        }
    
    def compute_eigenspace_constraint_loss(
        self,
        h_a: torch.Tensor,
        h_i: torch.Tensor,
        h_b: torch.Tensor
    ) -> torch.Tensor:
        """
        Encourage intermediate to handle hard directions optimally.
        
        Theory: For hard directions, I should be at the midpoint between A and B.
        For easy directions, I has more freedom.
        """
        # Project all embeddings onto eigenbasis
        proj_a = torch.einsum('ij,...j->...i', self.eigenvectors.T, h_a)
        proj_i = torch.einsum('ij,...j->...i', self.eigenvectors.T, h_i)
        proj_b = torch.einsum('ij,...j->...i', self.eigenvectors.T, h_b)
        
        # Midpoint in each direction
        midpoint = 0.5 * (proj_a + proj_b)
        
        # Deviation from midpoint, weighted by eigenvalue
        # (harder directions should be closer to midpoint)
        deviation = proj_i - midpoint
        weighted_deviation = self.eigenvalues * (deviation ** 2)
        
        # Only penalize hard directions (top n_hard_directions)
        hard_direction_loss = weighted_deviation[..., :self.n_hard_directions].sum(dim=-1)
        
        return hard_direction_loss.mean()
    
    def generate_steered_intermediate(
        self,
        mol_a: torch.Tensor,
        mol_b: torch.Tensor,
        n_candidates: int = 10,
        optimization_steps: int = 100,
        lr: float = 0.01
    ) -> Tuple[torch.Tensor, dict]:
        """
        Generate an optimal intermediate molecule using NTM steering.
        
        Args:
            mol_a, mol_b: Source and target molecules
            n_candidates: Number of random initializations
            optimization_steps: Gradient descent steps per candidate
            lr: Learning rate for latent optimization
        
        Returns:
            best_intermediate: Generated molecule with lowest path length
            info: Diagnostic information
        """
        # Encode endpoints
        with torch.no_grad():
            h_a = self.ntm_encoder(mol_a)
            h_b = self.ntm_encoder(mol_b)
            d_direct = self.compute_ntm_distance(h_a, h_b)
        
        best_z = None
        best_path_length = float('inf')
        all_results = []
        
        for candidate_idx in range(n_candidates):
            # Initialize latent code
            z = torch.randn(1, self.latent_dim, requires_grad=True)
            optimizer = torch.optim.Adam([z], lr=lr)
            
            for step in range(optimization_steps):
                optimizer.zero_grad()
                
                # Generate molecule and encode
                x_i = self.generator(z)
                h_i = self.ntm_encoder(x_i)
                
                # Path length loss
                d_ai = self.compute_ntm_distance(h_a, h_i)
                d_ib = self.compute_ntm_distance(h_i, h_b)
                path_loss = d_ai + d_ib
                
                # Eigenspace constraint
                eigen_loss = self.compute_eigenspace_constraint_loss(h_a, h_i, h_b)
                
                # Total loss
                total_loss = (self.steering_weight * path_loss + 
                             self.eigenspace_weight * eigen_loss)
                
                total_loss.backward()
                optimizer.step()
            
            # Evaluate final candidate
            with torch.no_grad():
                x_final = self.generator(z)
                h_final = self.ntm_encoder(x_final)
                final_path = self.compute_ntm_distance(h_a, h_final) + \
                            self.compute_ntm_distance(h_final, h_b)
                
                all_results.append({
                    'z': z.detach().clone(),
                    'path_length': final_path.item(),
                    'improvement': (d_direct - final_path).item() / d_direct.item()
                })
                
                if final_path < best_path_length:
                    best_path_length = final_path.item()
                    best_z = z.detach().clone()
        
        # Generate best intermediate
        with torch.no_grad():
            best_intermediate = self.generator(best_z)
            h_best = self.ntm_encoder(best_intermediate)
            
            # Decompose the transformation
            decomp_ai = self.decompose_transformation(h_a, h_best)
            decomp_ib = self.decompose_transformation(h_best, h_b)
        
        info = {
            'd_direct': d_direct.item(),
            'd_via_best': best_path_length,
            'improvement': (d_direct.item() - best_path_length) / d_direct.item(),
            'n_candidates': n_candidates,
            'all_results': all_results,
            'decomposition_a_to_i': decomp_ai,
            'decomposition_i_to_b': decomp_ib
        }
        
        return best_intermediate, info

print("NTMSteeredGenerator defined.")
print("\nKey features:")
print("  - Decomposes transformation difficulty by eigendirection")
print("  - Applies eigenspace constraints (hard directions → midpoint)")
print("  - Optimizes latent code using steering gradients")
print("  - Returns interpretable decomposition of generated path")

---
## Theoretical Summary: NTM Difficulty Components as Steering Signals

### The Key Insight

NTM provides more than a scalar difficulty prediction — it provides a **geometric decomposition** of transformation difficulty that can directly guide molecular generation.

### Difficulty Components Available for Steering

| Component | Mathematical Form | Steering Signal |
|-----------|------------------|-----------------|
| **Total difficulty** | $d_M(A,B) = \sqrt{\Delta h^T M \Delta h}$ | Scalar reward/loss |
| **Per-direction difficulty** | $d_i^2 = \lambda_i (v_i^T \Delta h)^2$ | Which directions to avoid |
| **Steering gradient** | $\nabla_h [d_M(A,h) + d_M(h,B)]$ | Direction to move in embedding space |
| **Easy subspace** | $\text{span}\{v_i : \lambda_i < \tau\}$ | Constraint for generation |
| **Hard direction midpoint** | $\frac{1}{2}(v_k^T h_A + v_k^T h_B)$ | Target for hard components |

### The Steering Principle (Formal Statement)

**Theorem (NTM Steering)**: Let $\mathbf{M} = \sum_i \lambda_i \mathbf{v}_i \mathbf{v}_i^T$ be the NTM metric tensor. For a transformation $A \to B$ with difficulty dominated by direction $k$ (i.e., $\lambda_k (v_k^T \Delta h)^2 \gg \lambda_j (v_j^T \Delta h)^2$ for $j \neq k$), the optimal intermediate $I^*$ satisfies:

1. **Hard direction constraint**: $\mathbf{v}_k^T \mathbf{h}_{I^*} \approx \frac{1}{2}(\mathbf{v}_k^T \mathbf{h}_A + \mathbf{v}_k^T \mathbf{h}_B)$

2. **Easy direction freedom**: $\mathbf{v}_j^T \mathbf{h}_{I^*}$ can vary freely for $\lambda_j \ll \lambda_k$

3. **Path length reduction**: $d_M(A, I^*) + d_M(I^*, B) \leq d_M(A, B)$ with equality iff $I^*$ lies on the geodesic

**Corollary**: A generative model steered by NTM gradients will naturally discover intermediates that:
- Split hard transformations into two easier steps
- Preserve freedom in easy directions (chemical diversity)
- Lie near the geodesic path in embedding space

### Connection to FEP Practice

| NTM Steering Concept | FEP Interpretation |
|---------------------|-------------------|
| Hard eigendirection | Core structural change (scaffold, charge) |
| Easy eigendirection | Peripheral modification (R-group) |
| Midpoint constraint | Intermediate has "half" of hard change |
| Path length reduction | Better λ-window overlap, faster convergence |

### Why This Is Novel

Previous approaches to intermediate generation:
- **LOMAP/Kartograf**: Use chemical similarity heuristics (not learned)
- **Fingerprint interpolation**: Operate in non-metric space
- **Random sampling + filtering**: No gradient signal

NTM steering provides:
- **Learned difficulty metric** from actual FEP outcomes
- **Differentiable gradients** for end-to-end optimization
- **Interpretable decomposition** via eigenstructure
- **Principled constraints** from Riemannian geometry

---
## Addressing the Embedding-to-Molecule Gap

### The Challenge

Steering operates in **continuous embedding space** $\mathbf{h}$, but we need **discrete valid molecules**. The optimal embedding $\mathbf{h}_{I^*}$ may not correspond to any real molecule.

### Three Scenarios

| Scenario | Description | Outcome |
|----------|-------------|---------|
| **Dense coverage** | Generator can produce molecules near any $\mathbf{h}$ | Steering works well |
| **Sparse coverage** | Large regions of embedding space have no valid molecules | May get stuck in invalid regions |
| **Misaligned spaces** | Generator's latent space doesn't align with NTM embedding | Gradients may be uninformative |

### Mitigation Strategies

**1. Validity-Constrained Optimization**

Add a validity term to the steering loss:

$$\mathcal{L}_{total} = \mathcal{L}_{path} + \lambda_{valid} \cdot \mathcal{L}_{validity}(G(z))$$

where $\mathcal{L}_{validity}$ penalizes chemically invalid structures (e.g., via a discriminator or rule-based checker).

**2. Projected Gradient Descent**

After each gradient step, project back onto the manifold of valid molecules:

$$z_{t+1} = \text{Project}_{\text{valid}}(z_t - \eta \nabla_z \mathcal{L}_{path})$$

**3. Discrete Search with Continuous Guidance**

Use steering gradient to *rank* discrete candidates rather than optimize continuously:

1. Generate candidate pool from generator
2. Score each by NTM path length
3. Use steering gradient direction to bias sampling toward promising regions

**4. Joint Training**

Train the generator and NTM together so their spaces align:

$$\mathcal{L}_{joint} = \mathcal{L}_{generation} + \mathcal{L}_{NTM} + \alpha \cdot \mathcal{L}_{alignment}$$

where $\mathcal{L}_{alignment}$ encourages the generator's latent space to be smooth w.r.t. NTM distance.

---
## Conditions for Effective Steering

### When Does NTM Steering Help?

Not all transformations benefit from intermediates. Here we formalize when steering is effective.

### Condition 1: High Path Curvature

Steering helps when the direct A→B path has high **curvature** in the NTM metric space. Curvature indicates the path crosses a difficulty barrier.

**Diagnostic**: Compute the Hessian of the energy landscape at the midpoint:

$$\kappa = \mathbf{n}^T \nabla^2 E \, \mathbf{n}$$

where $\mathbf{n}$ is the normal to the A→B direction. High $\kappa$ → barrier exists → intermediates help.

### Condition 2: Anisotropic Difficulty

Steering is most effective when difficulty is **concentrated in few eigendirections**:

$$\text{Anisotropy} = \frac{\lambda_{max}}{\lambda_{min}} \gg 1$$

If all eigenvalues are similar (isotropic), there's no "easy" direction to exploit.

### Condition 3: Reachable Intermediate Region

There must exist valid molecules in the low-difficulty region. Define reachability:

$$\mathcal{R}(A, B) = \{I : I \text{ is valid}, \, d_M(A,I) + d_M(I,B) < (1 + \epsilon) \cdot d_M(A,B)\}$$

Steering works if $|\mathcal{R}(A, B)| > 0$.

### Decision Criterion

**Use steering when**:
1. $d_M(A, B) > \tau_{high}$ (direct path is difficult)
2. Anisotropy ratio $> 2$ (exploitable easy directions exist)
3. Generator can reach the intermediate region

**Skip steering when**:
1. $d_M(A, B) < \tau_{low}$ (already easy)
2. Isotropic difficulty (no easy directions)
3. A and B are in same local basin (no barrier)

In [ ]:
# =============================================================================
# Demonstration: Steering in Action
# =============================================================================

def demonstrate_steering_process():
    """
    Demonstrate how NTM difficulty components steer generation.
    
    This simulation shows the optimization trajectory in embedding space.
    """
    np.random.seed(42)
    torch.manual_seed(42)
    
    # Setup: 8D embedding space with clear eigenstructure
    d = 8
    
    # Create metric with 2 hard directions, 6 easy directions
    eigenvalues = torch.tensor([5.0, 4.0, 0.5, 0.4, 0.3, 0.2, 0.1, 0.1])
    
    # Random orthogonal eigenvectors
    random_matrix = torch.randn(d, d)
    Q, _ = torch.linalg.qr(random_matrix)
    eigenvectors = Q
    
    M = eigenvectors @ torch.diag(eigenvalues) @ eigenvectors.T
    
    # Define A and B with significant hard-direction component
    h_A = torch.randn(d) * 0.5
    h_B = h_A.clone()
    
    # Add large component in hard directions
    h_B += 2.0 * eigenvectors[:, 0]  # Large change in hardest direction
    h_B += 1.5 * eigenvectors[:, 1]  # Moderate change in second hardest
    h_B += 0.3 * eigenvectors[:, 4]  # Small change in easy direction
    
    # Compute direct difficulty
    delta = h_B - h_A
    d_direct = torch.sqrt(delta @ M @ delta)
    
    # Decompose difficulty
    projections = eigenvectors.T @ delta
    difficulty_per_dir = eigenvalues * (projections ** 2)
    
    print("Transformation A → B Analysis")
    print("=" * 50)
    print(f"Total NTM difficulty: {d_direct.item():.3f}")
    print(f"\nDifficulty by eigendirection:")
    for i in range(d):
        pct = 100 * difficulty_per_dir[i] / difficulty_per_dir.sum()
        bar = "█" * int(pct / 5)
        hard_label = " (HARD)" if eigenvalues[i] > 1 else ""
        print(f"  v{i+1} (λ={eigenvalues[i]:.1f}){hard_label}: {pct:5.1f}% {bar}")
    
    # Simulate steering optimization
    print("\n" + "=" * 50)
    print("Steering Optimization Trajectory")
    print("=" * 50)
    
    # Initialize intermediate at random point
    h_I = torch.randn(d) * 0.3
    h_I.requires_grad_(True)
    
    optimizer = torch.optim.Adam([h_I], lr=0.1)
    
    trajectory = []
    
    for step in range(50):
        optimizer.zero_grad()
        
        # Path length
        d_ai = torch.sqrt((h_I - h_A) @ M @ (h_I - h_A) + 1e-8)
        d_ib = torch.sqrt((h_B - h_I) @ M @ (h_B - h_I) + 1e-8)
        path_length = d_ai + d_ib
        
        # Eigenspace constraint: hard directions should be at midpoint
        proj_I = eigenvectors.T @ h_I
        proj_A = eigenvectors.T @ h_A
        proj_B = eigenvectors.T @ h_B
        midpoint = 0.5 * (proj_A + proj_B)
        
        # Penalize deviation from midpoint in hard directions
        hard_deviation = eigenvalues[:2] @ ((proj_I[:2] - midpoint[:2]) ** 2)
        
        loss = path_length + 0.5 * hard_deviation
        
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            trajectory.append({
                'step': step,
                'path_length': path_length.item(),
                'd_ai': d_ai.item(),
                'd_ib': d_ib.item(),
                'improvement': (d_direct.item() - path_length.item()) / d_direct.item(),
                'hard_dir_position': proj_I[:2].clone().numpy()
            })
        
        if step % 10 == 0:
            imp = trajectory[-1]['improvement']
            print(f"  Step {step:2d}: path={path_length.item():.3f}, "
                  f"improvement={imp:+.1%}")
    
    # Final analysis
    print("\n" + "=" * 50)
    print("Final Result")
    print("=" * 50)
    
    with torch.no_grad():
        final_d_ai = torch.sqrt((h_I - h_A) @ M @ (h_I - h_A))
        final_d_ib = torch.sqrt((h_B - h_I) @ M @ (h_B - h_I))
        final_path = final_d_ai + final_d_ib
        
        print(f"Direct path:     {d_direct.item():.3f}")
        print(f"Via intermediate: {final_path.item():.3f}")
        print(f"  A → I: {final_d_ai.item():.3f}")
        print(f"  I → B: {final_d_ib.item():.3f}")
        print(f"Improvement: {(d_direct - final_path).item() / d_direct.item():.1%}")
        
        # Check hard direction positioning
        proj_I_final = eigenvectors.T @ h_I
        proj_midpoint = 0.5 * (eigenvectors.T @ h_A + eigenvectors.T @ h_B)
        
        print(f"\nHard direction analysis:")
        for i in range(2):
            print(f"  v{i+1}: I at {proj_I_final[i].item():.3f}, "
                  f"midpoint at {proj_midpoint[i].item():.3f}, "
                  f"deviation = {abs(proj_I_final[i] - proj_midpoint[i]).item():.3f}")
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Plot 1: Path length over optimization
    ax1 = axes[0]
    steps = [t['step'] for t in trajectory]
    path_lengths = [t['path_length'] for t in trajectory]
    ax1.plot(steps, path_lengths, 'b-', linewidth=2)
    ax1.axhline(d_direct.item(), color='r', linestyle='--', label='Direct path')
    ax1.set_xlabel('Optimization Step')
    ax1.set_ylabel('Path Length')
    ax1.set_title('Steering Optimization Progress')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Trajectory in hard direction subspace
    ax2 = axes[1]
    hard_positions = np.array([t['hard_dir_position'] for t in trajectory])
    
    # Plot trajectory
    ax2.plot(hard_positions[:, 0], hard_positions[:, 1], 'b-', alpha=0.5, linewidth=1)
    ax2.scatter(hard_positions[0, 0], hard_positions[0, 1], c='gray', s=100, 
                marker='o', label='Start', zorder=5)
    ax2.scatter(hard_positions[-1, 0], hard_positions[-1, 1], c='green', s=100, 
                marker='^', label='Final I', zorder=5)
    
    # Plot A, B, midpoint in hard subspace
    proj_A_np = (eigenvectors.T @ h_A)[:2].numpy()
    proj_B_np = (eigenvectors.T @ h_B)[:2].numpy()
    midpoint_np = 0.5 * (proj_A_np + proj_B_np)
    
    ax2.scatter(proj_A_np[0], proj_A_np[1], c='blue', s=150, marker='o', label='A', zorder=5)
    ax2.scatter(proj_B_np[0], proj_B_np[1], c='red', s=150, marker='s', label='B', zorder=5)
    ax2.scatter(midpoint_np[0], midpoint_np[1], c='orange', s=100, marker='*', 
                label='Midpoint', zorder=5)
    
    ax2.set_xlabel('Hard Direction v1')
    ax2.set_ylabel('Hard Direction v2')
    ax2.set_title('Trajectory in Hard Subspace')
    ax2.legend(loc='upper left', fontsize=8)
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Difficulty decomposition comparison
    ax3 = axes[2]
    
    with torch.no_grad():
        # Direct path decomposition
        direct_decomp = eigenvalues * (projections ** 2)
        
        # Via intermediate decomposition
        delta_ai = h_I - h_A
        delta_ib = h_B - h_I
        proj_ai = eigenvectors.T @ delta_ai
        proj_ib = eigenvectors.T @ delta_ib
        via_decomp = eigenvalues * (proj_ai ** 2) + eigenvalues * (proj_ib ** 2)
    
    x = np.arange(d)
    width = 0.35
    
    ax3.bar(x - width/2, direct_decomp.numpy(), width, label='Direct A→B', color='red', alpha=0.7)
    ax3.bar(x + width/2, via_decomp.numpy(), width, label='Via I', color='green', alpha=0.7)
    
    ax3.set_xlabel('Eigendirection')
    ax3.set_ylabel('Difficulty Contribution')
    ax3.set_title('Difficulty Decomposition')
    ax3.set_xticks(x)
    ax3.set_xticklabels([f'v{i+1}\n(λ={eigenvalues[i]:.1f})' for i in range(d)], fontsize=8)
    ax3.legend()
    ax3.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("\nKey Observations:")
    print("  1. Steering gradient moves I toward midpoint in hard directions")
    print("  2. Path length decreases as I approaches optimal position")
    print("  3. Hard direction contributions are split between A→I and I→B")
    print("  4. Easy directions contribute minimally to total difficulty")

demonstrate_steering_process()

---
## Future Directions

### 1. Multi-Step Paths
Extend to k > 1 intermediates: A → I₁ → I₂ → ... → B

### 2. Uncertainty-Aware Generation
Generate intermediates that also reduce prediction uncertainty

### 3. Retrosynthetic Integration
Couple with retrosynthesis models to ensure intermediates are makeable

### 4. Active Learning Loop
Run FEP on generated intermediates, update NTM, regenerate

### 5. Protein-Aware Generation
Condition generation on binding site to ensure intermediates bind

---

## Summary

| Concept | Description |
|---------|-------------|
| **Goal** | Generate intermediates that shorten thermodynamic path |
| **Theory** | Circumvent energy barriers via lower-energy routes |
| **Methods** | VAE interpolation, NTM-guided generation, diffusion |
| **Constraints** | Chemical validity, synthetic accessibility |
| **When to use** | High direct difficulty, significant improvement, feasible intermediate |

This extends NTM from a **predictive** tool to an **optimization** tool — not just predicting which transformations are hard, but actively finding ways to make them easier.